In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import string
import nltk
nltk.download ('stopwords')
from nltk.corpus import stopwords 
from nltk.tokenize import WordPunctTokenizer

from config import MAPBOX_API_KEY
import plotly
import plotly.offline as py
import plotly.graph_objs as go
import plotly.express as px

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/MariloyHJimenez/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
import warnings
warnings.filterwarnings('ignore')

# Importing Clean Data

In [3]:
file_path = "Resources/Final_philadelphia_reviews.csv"

phillies_df = pd.read_csv(file_path)
phillies_df["business_id"].count()

10000

In [4]:
yelp_reviews_df = phillies_df[['review_id', 'user_id', 'business_id', 'text', 
                               'stars_business', 'review_count']]

In [5]:
#restaurants_df = pd.read_csv('Resources/Philadelphia_businesses.csv')
restaurants_df = pd.read_csv("Resources/Philly_NearRestaurants.csv")
#Check Null values in restaurantes
restaurants_df.rename (columns={'ID':'Business_Id'}, inplace=True)
restaurants_df['Business_Id'].count()

50

In [6]:
restaurants_df.head(10)

,name,latitude,longitude,stars_business,categories,review_count,Business_Id
0,Saad's Halal Restaurant,39.954963,-75.211851,4.5,"Halal, Middle Eastern, Restaurants, Food, Indi...",702,6_LnAQQ0-mml8YgpfRjGuA
1,Mood Cafe,39.948664,-75.215832,4.5,"Cafes, Pakistani, Juice Bars & Smoothies, Rest...",458,U30ggGzFpXvc2NZYwOW3qg
2,48th Street Grille,39.953137,-75.218661,4.5,"Internet Cafes, Restaurants, Food, Seafood, Am...",275,KYPhGJIibu_7ePIX4HNs6A
3,Cucina Zapata,39.954289,-75.185538,4.5,"Thai, Food Stands, Restaurants, Mexican",248,B-DiQpcSTJ7oMMnwzbAGTQ
4,Fu-Wah Mini Market,39.948292,-75.217014,4.5,"Grocery, Food, Convenience Stores, Restaurants...",247,Biidj3QszVuVO0Q85g5NPw
5,Dottie's Donuts,39.948345,-75.213656,4.5,"Restaurants, Food, Vegan, Cafes, Donuts",206,A3Qt87F7ZaAwCW4CyyB1Fw
6,Tacos Don Memo,39.951988,-75.199123,4.5,"Food Trucks, Specialty Food, Food, Restaurants...",200,eXKblEHP3YJYU1Awz08hVw
7,Lee's Deli,39.948441,-75.217116,4.5,"Sandwiches, Burgers, Delis, Restaurants, Local...",175,KHe6HAqmyioITwTvg9Bbcg
8,White Dog Cafe,39.953558,-75.192905,4.0,"Cocktail Bars, Gluten-Free, Bars, Diners, Vege...",1301,ZKPrXH_GNW_AtZ31tP3NmA
9,Sabrina's Café,39.959879,-75.190681,4.0,"Breakfast & Brunch, Restaurants, Vegetarian, A...",833,6ewV-e7-39oqYUq3yZuIyw


In [7]:
# Fill with empty string the NaN reviews
yelp_reviews_df.dropna(inplace=True)
yelp_reviews_df[['text']] = yelp_reviews_df[['text']].fillna('')
yelp_reviews_df.head()

,review_id,user_id,business_id,text,stars_business,review_count
1,sFQrhTbTah0o2kU_Pi2D0Q,Tu4ATXLhy8kRTjpQCnl2pA,ytynqOUb3hjKeJfRj5Tshw,One;of;my;favorite;places;to;go;to;in;Philly;;...,4.5,5721
2,kqn1uP3LRVjVDUD44ZSu1A,vRNb2IaGlsZRA_wUf3Ov8w,ytynqOUb3hjKeJfRj5Tshw,This might be a bit unfair to have a single re...,4.5,5721
3,qMsTe9QznpNQk1AKbYLp-w,29K-usmZfVDeIaQ85EG54A,ytynqOUb3hjKeJfRj5Tshw,Alright; I remember the first time I went to t...,4.5,5721
4,nmMIRBNONIICe7CFHnfadQ,1jE--VcTddwXGampD23JCg,ytynqOUb3hjKeJfRj5Tshw,It's;an;experience;;to;say;the;least!;;Not;as;...,4.5,5721
6,dXVhYlcX9X9kqVI16AoBHg,lavyYLh68LxIBhJdIE5f_g,ytynqOUb3hjKeJfRj5Tshw,fresh;seafood;and;produce.;fresh;breads;;chees...,4.5,5721


In [8]:
yelp_reviews_df.count()

review_id         8303
user_id           8303
business_id       8303
text              8303
stars_business    8303
review_count      8303
dtype: int64

In [9]:
yelp_reviews_df.rename (columns={'review_id': 'Review_ID', 'user_id' :'User_Id', 
                        'business_id':'Business_Id', 'text':'Reviews', 
                'stars_business': 'Rating', 'review_count' :'Review_count'}, inplace=True)


# Begin the reviews cleaning, selecting only stars and text

In [10]:
yelp_reviews_df["Reviews"]

1       One;of;my;favorite;places;to;go;to;in;Philly;;...
2       This might be a bit unfair to have a single re...
3       Alright; I remember the first time I went to t...
4       It's;an;experience;;to;say;the;least!;;Not;as;...
6       fresh;seafood;and;produce.;fresh;breads;;chees...
                              ...                        
9993    Let down. I was in a hurry to catch my next fl...
9994    I;dont;give;one;star;reviews;but;they;earned;i...
9995    I;would;give;negative;stars;if;I;could.;Chewy;...
9997    So if you want a cheesesteak at PHL airport; t...
9999    Have;to;say;Geno's;is;a;great;place;to;get;a;c...
Name: Reviews, Length: 8303, dtype: object

In [11]:
#Select only stars and text
reviews_df = yelp_reviews_df[['Business_Id', 'User_Id', 'Rating', 'Reviews']]
reviews_df["Reviews"] = yelp_reviews_df["Reviews"].str.replace(";", " ")
reviews_df["Reviews"]

1       One of my favorite places to go to in Philly  ...
2       This might be a bit unfair to have a single re...
3       Alright  I remember the first time I went to t...
4       It's an experience  to say the least!  Not as ...
6       fresh seafood and produce. fresh breads  chees...
                              ...                        
9993    Let down. I was in a hurry to catch my next fl...
9994    I dont give one star reviews but they earned i...
9995    I would give negative stars if I could. Chewy ...
9997    So if you want a cheesesteak at PHL airport  t...
9999    Have to say Geno's is a great place to get a c...
Name: Reviews, Length: 8303, dtype: object

In [12]:
def text_process(mess):
    """
    Takes in a string of text, then performs the following:
    1. Remove all punctuation
    2. Remove all stopwords
    3. Returns a list of the cleaned text
    """
    print (mess)
    
   # Check characters to see if they are in punctuation          
    nopunc = [char for char in mess if char not in string.punctuation]

    # Join t('stop_word(he characters again to form the string.
    nopunc = ''.join(nopunc)
    
    # Now just remove any stopwords
    return " ".join([word for word in nopunc.split() if word.lower() not in stop])

In [13]:
from nltk.corpus import stopwords
stop = []

for word in stopwords.words('english'):
    s = [char for char in word if char not in string.punctuation]
    stop.append(''.join(s))

In [14]:
reviews_df['Reviews'] = reviews_df['Reviews'].apply(text_process)

One of my favorite places to go to in Philly  and I take every one of my visiting friends here! This place has a lot of yummy stores and awesome stores ( recommended: the grilled cheese place at Meltkraft Grilled Cheese  the donuts at Beilers  4 Seasons Juice Bar  Muellar's Chocolate Co.  icecream from Bassetts  and local honey from Bee Natural). I love coming here  and even though it can get really crowded during lunch hours on the weekends  it's still worth it.
This might be a bit unfair to have a single review  as each stall has very different fare with a variety of qualities. It can definitely be considered food heaven  but on the contrary on a Saturday morning can be Philadelphia hell. The produce at Iovines is very good and fresh  and very very reasonably priced. It can be hectic but the lines go extremely quickly. The rest of the market tends to have very long lines - but the lines can be an indication of the HAVE TO eat. The Original Turkeys turkey sandwhich with cranberry and 

The Han Dynasty was a great venue for our wedding rehearsal dinner! Their dishes are very authentic Chinese cooking and everything we had tasted great. The menu selection and service were excellent  and Megan helped us create a custom menu to mix authentic Chinese and kosher styles  while also incorporating plenty of items for vegetarian and other dietary restrictions. The restaurant comfortably seated 80+ people in their side area with constant  friendly service the whole night. In spite of some misunderstandings on menu totals and pricing  Megan was very patient and accommodating and helped us sort everything out. At the end of the night  our family and friends loved the food and everyone had a great time.  I highly recommend the Han Dynasty to anyone for great food  service  and events coordinating!
Han Dynasty is one of my favorite restaurants in Philadelphia.  I first found the Manayunk location  and my wife and I ate there several times.  We heard about it from a friend who turne

Southern Sangria is stronger than any European Sangria you've ever had. It's also more fruity  more flavorful and more maroon than any of that flimsy Spanish drink.  Prior to your meal  Ms. Tootsie serves up a giant cornbread muffin as big as your fist. Slather some strawberry honey butter on around the brown top and take a giant bite. The cornbread is as gold as the ripest corn and exudes a flavor akin to midwest state fair corn dunked in butter. It's moist  bountiful and hearty...just the way cornbread was intended to be...  The Chicken and Waffles is simply amazing. Usually not one for any sort of chicken  the buttermilk fry on these strips was too crispy  too clean  too golden not to gobble up. The waffles were fat and golden  and much like their tender counterparts  were smooth and moist on the inside. Served with pure syrup and any other requested condiments  this appetizer is enough for a meal.
This place was amazing! Ms Reni made the most amazing drinks and had such an amazing 

This place hits the spot.  I didn't try the coffee  but went with tea instead  so I can't comment on the coffee's goodness today.  What I can say is they place great tunes  there is plenty of space  the wifi doesn't poop out  and the decor & vibe of the place is perfect for me (kinda country  chill).  The egg sammy I ordered is delish  hot and cheesy.  I'm working here very productively  with a view of the horse stable out the window.  So I'll definitely be back here to work and chow down again.
I've never eaten here  so I can't speak for the food. But the coffee is delicious  the staff is always cheerful and very helpful and eager to recommend their favorites. Very nice ambiance for sitting and relaxing with a cup of coffee and a book -- nice cozy atmosphere for a city coffee shop.
Visited here last Sunday & really enjoyed the food  and live music!  The service was a little slow but the music def kept you distracted!  I ordered the chicken livers & gravy & a salad very yummy! (Asked f

I have heard and read great reviews from many people about Dante & Luigi's  I was not disappointed. Every entree we had was delicious  the apps. and desserts were just as good. I've had better meals  but not by much  and never at this price point.
Don't try to bring a young child here  because they don't have highchairs or booster seats for little kids. So much for families being welcome... as long as your kids are old enough to sit in regular chairs I guess. Was pretty irritating to be turned away after braving a pretty nasty rainstorm specifically to go here on our last night in Philadelphia.
I'm a spoiled New Yorker  what can I say?  When I heard this was the oldest Italian restaurant in Philly  I had high expectations. When I saw the prices  I had higher expectations. Expectations not met.  I had the spaghetti with bolognese sauce which was good but I have had so much better. I think the sauce was the most disappointing part of the meal.  The bread served in the beginning is fresh 

I am ... in Love.  Philly Love...that's what they call it?  This is the perfect cafe that has outside seating  without the hustle and bustle of cars driving by and wafting dust in your face.  You actually look into the park  rather than another old building.  This cafe serves a variety of organic coffees  wine (their house boxed...yes boxed..wine..is fantastic!)  cheeses  snacks  deli options and even a dinner setting.    If I had to choose to work remotely  I would choose this little knook.  Outside seating at Taluluas or indoors on their cocktail tables.    Must tries: Iced Coffee Cappuccino Cheese Board  Salami Deli options  Cookie (free from yelp checkin!) House white wine I hear you can order cocktails from Talula's garden  but I'm so happy w/ the offerings here.    The service is great and don't forget the coffee punch card!  Just the absolute perfect setting and likely more affordable pricing than Talula's Garden.
Many legitimate complaints about wait times and food sitting out 

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



In [15]:
#Split train test for testing the model later
vld_size=0.15
X_train, X_valid, y_train, y_valid = train_test_split(reviews_df['Reviews'], yelp_reviews_df['Business_Id'], test_size = vld_size) 


# Create two tables of user, text and bussiness

In [16]:
reviews_df['Business_Id'].count()

8303

In [17]:
newreviews_df = reviews_df[reviews_df['Business_Id'].isin(restaurants_df["Business_Id"])]
newreviews_df['Business_Id'].count()

415

In [18]:
newreviews_df.head(25)

,Business_Id,User_Id,Rating,Reviews
391,ZKPrXH_GNW_AtZ31tP3NmA,uIHI_BzJcNMxFAZA5bYeXg,4.0,Recently visited dinner daughter visiting Univ...
392,ZKPrXH_GNW_AtZ31tP3NmA,s9FrHlk9kPnfSAdAgKNtWA,4.0,love place 2 reasons 1 husband proposed 2 amaz...
393,ZKPrXH_GNW_AtZ31tP3NmA,eFXy9H9H2tBfnyNahNUUEQ,4.0,Gotta love White Dog food service reliably goo...
394,ZKPrXH_GNW_AtZ31tP3NmA,ETyduRMWNWg7cjoZaRxmaQ,4.0,got chicken sandwich broccoli rabe side salad ...
395,ZKPrXH_GNW_AtZ31tP3NmA,zCDU0cm2K_R7HozWqTgm0w,4.0,Mold hot dog ordered mold bun Im paying 750 ho...
397,ZKPrXH_GNW_AtZ31tP3NmA,d2SmmiFbMXUxT5-cTzu0OQ,4.0,happened White Dog Cafe First time friend Dans...
398,ZKPrXH_GNW_AtZ31tP3NmA,nBdv5Br4G8pDWXXqSXhUug,4.0,Hello Last night wonderful meal tuna tartar du...
399,ZKPrXH_GNW_AtZ31tP3NmA,T9aisv9FFuEbO9hSgi-efw,4.0,wonderful little spot University City offers c...
611,kVTHnhgYUw-Pmr7wgDB4_g,uIMcOM39MNJfkEFVDWu2-A,3.5,Seating horrible came without reservation half...
612,kVTHnhgYUw-Pmr7wgDB4_g,YnbiDyzD3NpBTxeMQdtzDA,3.5,heard great things Distrito dying go Overall g...


In [19]:
userid_df = newreviews_df[['User_Id','Reviews']]
business_df = newreviews_df[['Business_Id', 'Reviews']]

In [20]:
userid_df.head()

,User_Id,Reviews
391,uIHI_BzJcNMxFAZA5bYeXg,Recently visited dinner daughter visiting Univ...
392,s9FrHlk9kPnfSAdAgKNtWA,love place 2 reasons 1 husband proposed 2 amaz...
393,eFXy9H9H2tBfnyNahNUUEQ,Gotta love White Dog food service reliably goo...
394,ETyduRMWNWg7cjoZaRxmaQ,got chicken sandwich broccoli rabe side salad ...
395,zCDU0cm2K_R7HozWqTgm0w,Mold hot dog ordered mold bun Im paying 750 ho...


In [21]:
business_df.head()

,Business_Id,Reviews
391,ZKPrXH_GNW_AtZ31tP3NmA,Recently visited dinner daughter visiting Univ...
392,ZKPrXH_GNW_AtZ31tP3NmA,love place 2 reasons 1 husband proposed 2 amaz...
393,ZKPrXH_GNW_AtZ31tP3NmA,Gotta love White Dog food service reliably goo...
394,ZKPrXH_GNW_AtZ31tP3NmA,got chicken sandwich broccoli rabe side salad ...
395,ZKPrXH_GNW_AtZ31tP3NmA,Mold hot dog ordered mold bun Im paying 750 ho...


In [22]:
userid_df = userid_df.groupby('User_Id').agg({'Reviews': ' '.join})
business_df = business_df.groupby('Business_Id').agg({'Reviews': ' '.join})

In [23]:
userid_df.head()

,Reviews
User_Id,
-5QrmUZTvniwryx3l3JM8g,favorite trendy ramen spot comfortable atmosph...
02EPgkmrh39Dr7uXc1bqTQ,Food excellent cheap quality quantity waitress...
0DB3Irpf_ETVXu_Ou9vPow,used live right next Desi Chaat House pretty m...
0YmEeQEzqwMwuWKvf3PTNA,got Philly STARVING Yelp place eat Tacos Memo ...
0dcknuP3VLaO_Q6ugNmLQA,eat coin aint cheap business better believe fa...


# User Tfidf Vectorizer with 5000 Features (represent 88% of all words)

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [25]:
#userid vectorizer
userid_vectorizer = TfidfVectorizer(tokenizer = WordPunctTokenizer().tokenize, max_features=5000)
userid_vectors = userid_vectorizer.fit_transform(userid_df['Reviews'])
userid_vectors.shape

(387, 4786)

In [26]:
userid_vectors

<387x4786 sparse matrix of type '<class 'numpy.float64'>'
	with 19738 stored elements in Compressed Sparse Row format>

In [27]:
#Business id vectorizer
businessid_vectorizer = TfidfVectorizer(tokenizer = WordPunctTokenizer().tokenize, max_features=5000)
businessid_vectors = businessid_vectorizer.fit_transform(business_df['Reviews'])
businessid_vectors.shape

(50, 4786)

# Matrix Factorization

In [28]:
userid_rating_matrix = pd.pivot_table(newreviews_df, values='Rating', index=['User_Id'], columns=['Business_Id'])
userid_rating_matrix.shape

(387, 50)

In [29]:
userid_rating_matrix.head()

Business_Id,-5Rah4ZvWsDu4oilUZxhtw,14ZGwnDyydXdSBsLXpSUrA,3gVSrS4kffGGZT8oXHsIcw,41LEg4ZgFDVnZfToxGXiGA,5ICrDkwtX4ykKOLVJ1tFJA,6GXMHrB8u-3nq87zE1Av0w,6_LnAQQ0-mml8YgpfRjGuA,6ewV-e7-39oqYUq3yZuIyw,8yGZhBwFFLtOVLY9Do8ohQ,99e7bysta1myyrQogFEWUQ,...,lS42krxXMm-HIk7dntRsKQ,pgri9CUZXGy9hi6UMMQR6A,qtyNbCXut-RQnnEQNJ9UzA,s2JiYzE7i68cXIV6YEcVTw,uB12vX5p9T8z0g2Eq3bBBQ,uE40984_YDgVvPeRpFcCaQ,uvj_tGQrGDSVXjNU4pHjCA,yG5tkMER9Ftkv4RDwMaQuA,zU5PPd9j4bYC_MepNSEDcA,ziyXteCmNmRprs10buP2iQ
User_Id,,,,,,,,,,,,,,,,,,,,,
-5QrmUZTvniwryx3l3JM8g,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
02EPgkmrh39Dr7uXc1bqTQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
0DB3Irpf_ETVXu_Ou9vPow,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0YmEeQEzqwMwuWKvf3PTNA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0dcknuP3VLaO_Q6ugNmLQA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
P = pd.DataFrame(userid_vectors.toarray(), index=userid_df.index, columns=userid_vectorizer.get_feature_names())
Q = pd.DataFrame(businessid_vectors.toarray(), index=business_df.index, columns=businessid_vectorizer.get_feature_names())


In [31]:
Q.head()

,0,030503062010,04,05,08,09,1,10,100,1012,...,étaient,、,。,不喜欢的人会很不喜欢,可以提前跟店家说一下不要加,唯一的缺憾是冰粉上面会加橙皮以及其他一些类似五仁的材料,红油,食物非常正宗,麻酱和红糖用得都很正,，
Business_Id,,,,,,,,,,,,,,,,,,,,,
-5Rah4ZvWsDu4oilUZxhtw,0.0,0.0,0.000000,0.037209,0.000000,0.0,0.0,0.02148,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14ZGwnDyydXdSBsLXpSUrA,0.0,0.0,0.058513,0.000000,0.058513,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3gVSrS4kffGGZT8oXHsIcw,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41LEg4ZgFDVnZfToxGXiGA,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5ICrDkwtX4ykKOLVJ1tFJA,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.00000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Gradient Decent Optimization

In [32]:
def matrix_factorization(R, P, Q, steps=25, gamma=0.001,lamda=0.02):
    for step in range(steps):
        for i in R.index:
            for j in R.columns:
                if R.loc[i,j]>0:
                    eij=R.loc[i,j]-np.dot(P.loc[i],Q.loc[j])
                    P.loc[i]=P.loc[i]+gamma*(eij*Q.loc[j]-lamda*P.loc[i])
                    Q.loc[j]=Q.loc[j]+gamma*(eij*P.loc[i]-lamda*Q.loc[j])
        e=0
        for i in R.index:
            for j in R.columns:
                if R.loc[i,j]>0:
                    e= e + pow(R.loc[i,j]-np.dot(P.loc[i],Q.loc[j]),2)+lamda*(pow(np.linalg.norm(P.loc[i]),2)+pow(np.linalg.norm(Q.loc[j]),2))
        if e<0.001:
            break
        
    return P,Q

In [33]:
%%time
P, Q = matrix_factorization(userid_rating_matrix, P, Q, steps=25, gamma=0.001,lamda=0.02)

CPU times: user 1min 20s, sys: 5.05 s, total: 1min 25s
Wall time: 30.2 s


In [34]:
Q.head()

,0,030503062010,04,05,08,09,1,10,100,1012,...,étaient,、,。,不喜欢的人会很不喜欢,可以提前跟店家说一下不要加,唯一的缺憾是冰粉上面会加橙皮以及其他一些类似五仁的材料,红油,食物非常正宗,麻酱和红糖用得都很正,，
Business_Id,,,,,,,,,,,,,,,,,,,,,
-5Rah4ZvWsDu4oilUZxhtw,0.000000e+00,0.000000e+00,0.000000e+00,0.042769,0.000000e+00,0.0,0.000000e+00,3.229334e-02,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14ZGwnDyydXdSBsLXpSUrA,9.174749e-12,5.411205e-08,9.292148e-02,0.000000,9.292148e-02,0.0,8.811944e-12,6.898484e-03,1.308731e-15,1.359473e-11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3gVSrS4kffGGZT8oXHsIcw,6.377822e-08,4.802993e-08,2.348853e-11,0.000000,2.348853e-11,0.0,6.203674e-08,9.758189e-08,1.688802e-11,9.468977e-08,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41LEg4ZgFDVnZfToxGXiGA,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5ICrDkwtX4ykKOLVJ1tFJA,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0,3.342382e-04,5.173488e-03,0.000000e+00,0.000000e+00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [35]:
# Here we can view the principal words that are mentioned about the restaurant.
Q.iloc[7].sort_values(ascending=False).head(10)

brunch       0.241662
toast        0.211703
eggs         0.183513
french       0.176270
fries        0.174674
challah      0.174463
app          0.173930
olives       0.170844
pineapple    0.170172
came         0.161119
Name: 6ewV-e7-39oqYUq3yZuIyw, dtype: float64

In [36]:
# Store P, Q and vectorizer in pickle file
import pickle
output = open('yelp_recommendation_model_8.pkl', 'wb')
pickle.dump(P,output)
pickle.dump(Q,output)
pickle.dump(userid_vectorizer,output)
output.close()

# Run prediction according User's preference

In [41]:
words = "burger fries"
test_df= pd.DataFrame([words], columns=['Reviews'])
test_df['Reviews'] = test_df['Reviews'].apply(text_process)
test_vectors = userid_vectorizer.transform(test_df['Reviews'])
test_v_df = pd.DataFrame(test_vectors.toarray(), index=test_df.index, columns=userid_vectorizer.get_feature_names())

predictItemRating=pd.DataFrame(np.dot(test_v_df.loc[0],Q.T),index=Q.index,columns=['Rating'])
foundRestaurants=pd.DataFrame.sort_values(predictItemRating,['Rating'],ascending=[0])[:7]

for i in foundRestaurants.index:
    name = restaurants_df[restaurants_df['Business_Id']==i]['name'].iloc[0]
    categories =restaurants_df[restaurants_df['Business_Id']==i]['categories'].iloc[0]
    latitude = restaurants_df[restaurants_df['Business_Id']==i]['latitude'].iloc[0]
    longitude = restaurants_df[restaurants_df['Business_Id']==i]['longitude'].iloc[0]
    rating = str(restaurants_df[restaurants_df['Business_Id']==i]['stars_business'].iloc[0])
    topRecommendations = pd.DataFrame({
        "Name": [name],
        "Categorie": [categories],
        "Latitude": [latitude],
        "Longitude": [longitude],
        "Rating": [rating]
    })

    print(restaurants_df[restaurants_df['Business_Id']==i]['name'].iloc[0])
    print(restaurants_df[restaurants_df['Business_Id']==i]['categories'].iloc[0])
    print(restaurants_df[restaurants_df['Business_Id']==i]['latitude'].iloc[0])
    print(restaurants_df[restaurants_df['Business_Id']==i]['longitude'].iloc[0])
    print(str(restaurants_df[restaurants_df['Business_Id']==i]['stars_business'].iloc[0])+ ' ' +str(restaurants_df[restaurants_df['Business_Id']==i]['review_count'].iloc[0]))
    
    print('')

#topRecommendations.head()

burger fries
Bobby's Burger Palace
Sandwiches, American (Traditional), Burgers, Restaurants, Salad
39.954086811
-75.2009592271
3.5 788

Lee's Deli
Sandwiches, Burgers, Delis, Restaurants, Local Flavor
39.9484414
-75.2171163
4.5 175

HipCityVeg - University City
Sandwiches, Food, Burgers, Restaurants, Vegan, Desserts
39.953549
-75.202991
4.0 314

New Deck Tavern
Irish, Gastropubs, Pubs, American (Traditional), Nightlife, Bars, Restaurants
39.9535171
-75.1926364399
3.5 419

Sabrina's Café
Breakfast & Brunch, Restaurants, Vegetarian, American (New), American (Traditional), Comfort Food, Cafes
39.9598792
-75.1906815
4.0 833

Hummus Grill
Specialty Food, Food, Sandwiches, Middle Eastern, Vegetarian, Ethnic Food, Restaurants, Mediterranean, Greek, Salad
39.954349
-75.2021586
4.0 372

Renata's Kitchen
Breakfast & Brunch, Caterers, Tapas/Small Plates, Event Planning & Services, Middle Eastern, Mediterranean, Restaurants, Greek
39.9496602
-75.2026672
4.0 203



In [38]:
px.set_mapbox_access_token(MAPBOX_API_KEY)
fig = px.scatter_mapbox(topRecommendations, lat="Latitude", 
                  lon="Longitude", color="Rating", hover_name="Name", size_max=30, zoom= 13, width=1200, height=800)

In [39]:
px.set_mapbox_access_token(MAPBOX_API_KEY)
px.scatter_mapbox(topRecommendations, lat="Latitude", 
                  lon="Longitude", color="Rating",hover_data=["Name","Latitude","Longitude"],  zoom= 10, width=1200, height=800)